![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Agents Lab Notebook v1.0.0
This notebook contains steps and code to demonstrate the use of agents
configured in Agent Lab in watsonx.ai. It introduces Python API commands
for authentication using API key and invoking a LangGraph agent with a watsonx chat model.

**Note:** Notebook code generated using Agent Lab will execute successfully.
If code is modified or reordered, there is no guarantee it will successfully execute.
For details, see: <a href="/docs/content/wsj/analyze-data/fm-prompt-save.html?context=wx" target="_blank">Saving your work in Agent Lab as a notebook.</a>

Some familiarity with Python is helpful. This notebook uses Python 3.12.

## Notebook goals
The learning goals of this notebook are:

* Defining a Python function for obtaining credentials from the IBM Cloud personal API key
* Creating an agent with a set of tools using a specified model and parameters
* Invoking the agent to generate a response 

# Setup

In [ ]:
# import dependencies
from langchain_ibm import ChatWatsonx
from ibm_watsonx_ai import APIClient
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from ibm_watsonx_ai.foundation_models.utils import Tool, Toolkit
import json
import requests

## watsonx API connection
This cell defines the credentials required to work with watsonx API for Foundation
Model inferencing.

**Action:** Provide the IBM Cloud personal API key. For details, see
<a href="https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui" target="_blank">documentation</a>.


In [ ]:
import os
import getpass

def get_credentials():
	return {
		"url" : "https://au-syd.ml.cloud.ibm.com",
		"apikey" : getpass.getpass("Please enter your api key (hit enter): ")
	}

def get_bearer_token():
    url = "https://iam.cloud.ibm.com/identity/token"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = f"grant_type=urn:ibm:params:oauth:grant-type:apikey&apikey={credentials['apikey']}"

    response = requests.post(url, headers=headers, data=data)
    return response.json().get("access_token")

credentials = get_credentials()

# Using the agent
These cells demonstrate how to create and invoke the agent
with the selected models, tools, and parameters.

## Defining the model id
We need to specify model id that will be used for inferencing:

In [ ]:
model_id = "meta-llama/llama-3-3-70b-instruct"

## Defining the model parameters
We need to provide a set of model parameters that will influence the
result:

In [ ]:
parameters = {
    "frequency_penalty": 0,
    "max_tokens": 2000,
    "presence_penalty": 0,
    "temperature": 0,
    "top_p": 1
}

## Defining the project id or space id
The API requires project id or space id that provides the context for the call. We will obtain
the id from the project or space in which this notebook runs:

In [ ]:
project_id = os.getenv("PROJECT_ID")
space_id = os.getenv("SPACE_ID")


## Creating the agent
We need to create the agent using the properties we defined so far:

In [ ]:
client = APIClient(credentials=credentials, project_id=project_id, space_id=space_id)

# Create the chat model
def create_chat_model():
    chat_model = ChatWatsonx(
        model_id=model_id,
        url=credentials["url"],
        space_id=space_id,
        project_id=project_id,
        params=parameters,
        watsonx_client=client,
    )
    return chat_model

In [ ]:
from ibm_watsonx_ai.deployments import RuntimeContext

context = RuntimeContext(api_client=client)


vector_index_id = "019ea5f7-cf1a-707c-8433-335c529f3c14"

def create_rag_tool(vector_index_id, api_client):
    config = {
        "vectorIndexId": vector_index_id,
        "projectId": project_id
    }

    tool_description = "Search information in documents to provide context to a user query. Useful when asked to ground the answer in specific knowledge about Digital Financial Literacy Knowledge Base"
    
    return create_utility_agent_tool("RAGQuery", config, api_client, tool_description=tool_description)



def create_utility_agent_tool(tool_name, params, api_client, **kwargs):
    from langchain_core.tools import StructuredTool
    utility_agent_tool = Toolkit(
        api_client=api_client
    ).get_tool(tool_name)

    tool_description = utility_agent_tool.get("description")

    if (kwargs.get("tool_description")):
        tool_description = kwargs.get("tool_description")
    elif (utility_agent_tool.get("agent_description")):
        tool_description = utility_agent_tool.get("agent_description")
    
    tool_schema = utility_agent_tool.get("input_schema")
    if (tool_schema == None):
        tool_schema = {
            "type": "object",
            "additionalProperties": False,
            "$schema": "http://json-schema.org/draft-07/schema#",
            "properties": {
                "input": {
                    "description": "input for the tool",
                    "type": "string"
                }
            }
        }
    
    def run_tool(**tool_input):
        query = tool_input
        if (utility_agent_tool.get("input_schema") == None):
            query = tool_input.get("input")

        results = utility_agent_tool.run(
            input=query,
            config=params
        )
        
        return results.get("output")
    
    return StructuredTool(
        name=tool_name,
        description = tool_description,
        func=run_tool,
        args_schema=tool_schema
    )


def create_custom_tool(tool_name, tool_description, tool_code, tool_schema, tool_params):
    from langchain_core.tools import StructuredTool
    import ast

    def call_tool(**kwargs):
        tree = ast.parse(tool_code, mode="exec")
        custom_tool_functions = [ x for x in tree.body if isinstance(x, ast.FunctionDef) ]
        function_name = custom_tool_functions[0].name
        compiled_code = compile(tree, 'custom_tool', 'exec')
        namespace = tool_params if tool_params else {}
        exec(compiled_code, namespace)
        return namespace[function_name](**kwargs)
        
    tool = StructuredTool(
        name=tool_name,
        description = tool_description,
        func=call_tool,
        args_schema=tool_schema
    )
    return tool

def create_custom_tools():
    custom_tools = []


def create_tools(context):
    tools = []
    tools.append(create_rag_tool(vector_index_id, client))
    
    config = None
    tools.append(create_utility_agent_tool("GoogleSearch", config, client))
    config = {
    }
    tools.append(create_utility_agent_tool("DuckDuckGo", config, client))
    config = {
        "maxResults": 5
    }
    tools.append(create_utility_agent_tool("Wikipedia", config, client))
    config = {
    }
    tools.append(create_utility_agent_tool("WebCrawler", config, client))

    return tools

In [ ]:
def create_agent(context):
    # Initialize the agent
    chat_model = create_chat_model()
    tools = create_tools(context)

    memory = MemorySaver()
    instructions = """# Additional Guidelines

* Use clear, simple, and beginner-friendly language when explaining financial concepts.
* Structure responses using headings, bullet points, numbered steps, and examples whenever helpful.
* Explain financial terms such as UPI, interest rates, EMIs, credit scores, insurance, investments, and budgeting in a way that non-experts can understand.
* When providing financial guidance, focus on education and awareness rather than personalized financial advice.
* Encourage users to verify important financial information with official banks, financial institutions, or government sources.
* Always prioritize user safety and digital security.
* Never request, store, or encourage sharing of sensitive information such as passwords, PINs, OTPs, CVV numbers, bank account credentials, Aadhaar numbers, or card details.
* If a user reports a suspicious transaction, scam, phishing attempt, or cybercrime, immediately recommend contacting their bank and reporting the incident through official channels.

# Cyber Safety Support

For fraud, scam, phishing, identity theft, or unauthorized transaction-related queries:

* Advise users to contact their bank immediately.
* Recommend reporting the incident through:

  * National Cyber Crime Reporting Portal: https://cybercrime.gov.in
  * Cyber Crime Helpline: 1930
* Provide practical steps to secure accounts, change passwords, and prevent further losses.

# Response Quality Standards

* Provide complete and accurate explanations.
* Present both benefits and risks when discussing financial products or services.
* Use real-world examples whenever appropriate.
* If information is uncertain or unavailable, clearly state the limitation instead of guessing.
* Maintain a professional, friendly, and supportive tone.
* Adapt explanations according to the user's level of financial knowledge.

# Scope of Assistance

You can help users with:

* UPI and digital payments
* Online banking and mobile banking
* Personal finance and budgeting
* Savings and financial planning
* Loans and interest rates
* Credit and debit cards
* Insurance basics
* Digital financial safety
* Fraud and scam awareness
* Government financial schemes
* General financial literacy and awareness

# Goal

Your primary objective is to improve digital financial literacy, promote safe financial practices, protect users from fraud, and empower them to make informed financial decisions with confidence.

You are FinAgent, an AI assistant dedicated to improving digital financial literacy and helping users make informed financial decisions safely and confidently.

When greeted with messages such as \"Hi\", \"Hello\", \"Hey\", or similar greetings, respond exactly with:

\"Hi, I am FinAgent. How can I help you?\"

Your responsibilities include:

* Educating users about digital payments, UPI, online banking, mobile wallets, credit and debit cards, savings, investments, loans, insurance, budgeting, and personal finance.
* Explaining financial concepts in simple, clear, and easy-to-understand language.
* Providing step-by-step guidance for financial processes and digital transactions.
* Promoting safe online financial practices and awareness of cyber frauds, phishing scams, identity theft, fake loan offers, investment scams, and other financial crimes.
* Encouraging users to verify information before making financial decisions.
* Offering practical tips and best practices for managing money, protecting financial accounts, and improving financial well-being.
* Supporting users from diverse backgrounds with patient, non-judgmental, and culturally inclusive responses.
* Adapting explanations to the user's level of financial knowledge, from beginners to advanced users.

Guidelines:

* Always provide accurate, balanced, and educational information.
* Use a professional, friendly, and supportive tone.
* Structure responses clearly using headings, bullet points, and examples when appropriate.
* When discussing risks, clearly explain both benefits and potential drawbacks.
* Never encourage users to share sensitive information such as passwords, PINs, OTPs, CVV numbers, bank account credentials, or personal financial details.
* Remind users to contact their bank or relevant authorities immediately if they suspect fraud or unauthorized transactions.
* When relevant, provide official resources and trusted links for reporting cybercrime, financial fraud, or obtaining financial assistance.

For cybercrime, fraud, or scam-related queries, recommend official resources such as:

* National Cyber Crime Reporting Portal (India): https://cybercrime.gov.in
* Cyber Crime Helpline: 1930
* Official websites of banks, financial institutions, and government agencies.

Your goal is to empower users with financial knowledge, help them avoid fraud, and build confidence in using digital financial services safely and responsibly.
"""

    agent = create_react_agent(chat_model, tools=tools, checkpointer=memory, prompt=instructions)

    return agent

In [ ]:
# Visualize the graph
from IPython.display import Image, display
from langchain_core.runnables.graph import CurveStyle, MermaidDrawMethod, NodeStyles

Image(
    create_agent(context).get_graph().draw_mermaid_png(
        draw_method=MermaidDrawMethod.API,
    )
)


## Invoking the agent
Let us now use the created agent, pair it with the input, and generate the response to your question:


In [ ]:
agent = create_agent(context)

def convert_messages(messages):
    converted_messages = []
    for message in messages:
        if (message["role"] == "user"):
            converted_messages.append(HumanMessage(content=message["content"]))
        elif (message["role"] == "assistant"):
            converted_messages.append(AIMessage(content=message["content"]))
    return converted_messages

question = input("Question: ")

messages = [{
    "role": "user",
    "content": question
}]

generated_response = agent.invoke(
    { "messages": convert_messages(messages) },
    { "configurable": { "thread_id": "42" } }
)

print_full_response = False

if (print_full_response):
    print(generated_response)
else:
    result = generated_response["messages"][-1].content
    print(f"Agent: {result}")


# Next steps
You successfully completed this notebook! You learned how to use
watsonx.ai inferencing SDK to generate response from the foundation model
based on the provided input, model id and model parameters. Check out the
official watsonx.ai site for more samples, tutorials, documentation, how-tos, and blog posts.

<a id="copyrights"></a>
### Copyrights

Licensed Materials - Copyright © 2026 IBM. This notebook and its source code are released under the terms of the ILAN License.
Use, duplication disclosure restricted by GSA ADP Schedule Contract with IBM Corp.

**Note:** The auto-generated notebooks are subject to the International License Agreement for Non-Warranted Programs (or equivalent) and License Information document for watsonx.ai Auto-generated Notebook (License Terms), such agreements located in the link below. Specifically, the Source Components and Sample Materials clause included in the License Information document for watsonx.ai Studio Auto-generated Notebook applies to the auto-generated notebooks.  

By downloading, copying, accessing, or otherwise using the materials, you agree to the <a href="https://www14.software.ibm.com/cgi-bin/weblap/lap.pl?li_formnum=L-AMCU-BYC7LF" target="_blank">License Terms</a>  